# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [12]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Coordenadas (fila, columna) leídas de la imagen del enunciado.
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,  # entrega
            (2, 2): 2.0,   # estación de carga
            (3, 5): -10.0, # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9
        self.normal_forward_prob = 0.90
        self.slippery_forward_prob = 0.60

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        return (
            0 <= row < self.height
            and 0 <= col < self.width
            and state not in self.walls
        )

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if not self.is_valid_state(state):
            raise ValueError(f"Estado inválido: {state}")
        if state in self.terminal_states:
            return self.terminal_states[state]
        return self.danger_states.get(state, self.living_reward)

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if not self.is_valid_state(state):
            raise ValueError(f"Estado inválido: {state}")
        if action not in self.actions:
            raise ValueError(f"Acción inválida: {action}")
        if self.is_terminal(state):
            return [(state, 1.0)]

        forward_prob = (
            self.slippery_forward_prob
            if state in self.slippery_states
            else self.normal_forward_prob
        )
        side_prob = (1.0 - forward_prob) / 2.0
        dr, dc = action
        candidates = [
            ((dr, dc), forward_prob),
            ((-dc, dr), side_prob),  # giro a la izquierda
            ((dc, -dr), side_prob),  # giro a la derecha
        ]

        # Acumular resultados repetidos (por ejemplo, dos choques contra paredes).
        probabilities = {}
        for (move_row, move_col), probability in candidates:
            next_state = (state[0] + move_row, state[1] + move_col)
            if not self.is_valid_state(next_state):
                next_state = state
            probabilities[next_state] = probabilities.get(next_state, 0.0) + probability

        return list(probabilities.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [13]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [14]:
def expected_next_value(grid, state, action, V):
    return sum(
        probability * V[next_state]
        for next_state, probability in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    states = grid.states()
    V = {state: 0.0 for state in states}

    for iteration in range(1, max_iter + 1):
        previous = V.copy()
        delta = 0.0
        for state in states:
            if grid.is_terminal(state):
                V[state] = grid.get_reward(state)
            else:
                best_next = max(
                    expected_next_value(grid, state, action, previous)
                    for action in grid.actions
                )
                V[state] = grid.get_reward(state) + grid.gamma * best_next
            delta = max(delta, abs(V[state] - previous[state]))
        if delta < threshold:
            return V, iteration

    raise RuntimeError(f"Value Iteration no convergió en {max_iter} iteraciones")


def extract_policy(grid, V):
    return {
        state: max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V),
        )
        for state in grid.states()
        if not grid.is_terminal(state)
    }



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [15]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    states = grid.states()
    V = {state: 0.0 for state in states}

    for _ in range(max_iter):
        previous = V.copy()
        delta = 0.0
        for state in states:
            if grid.is_terminal(state):
                V[state] = grid.get_reward(state)
            else:
                V[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(
                        grid, state, policy[state], previous
                    )
                )
            delta = max(delta, abs(V[state] - previous[state]))
        if delta < threshold:
            return V

    raise RuntimeError(f"Policy Evaluation no convergió en {max_iter} iteraciones")


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for iteration in range(1, max_iter + 1):
        V = policy_evaluation(grid, policy, threshold=threshold)
        improved_policy = policy_improvement(grid, V)
        changes = sum(
            policy[state] != improved_policy[state] for state in policy
        )
        history.append({"iteration": iteration, "policy_changes": changes})
        policy = improved_policy
        if changes == 0:
            return policy, V, history

    raise RuntimeError(f"Policy Iteration no convergió en {max_iter} iteraciones")



## Parte 4 — Visualización y comparación


In [16]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [17]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'iteration': 1, 'policy_changes': 17}, {'iteration': 2, 'policy_changes': 5}, {'iteration': 3, 'policy_changes': 1}, {'iteration': 4, 'policy_changes': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.83


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas

1. Con los parámetros originales, desde `START` la política se dirige a la **estación de carga +2**: la ruta intencional es `(0,0) → (0,1) → (0,2) → (1,2) → (2,2)`.
2. Una recompensa menor puede ser óptima si está más cerca y exige menos pasos o menos riesgo. Aquí, el costo acumulado por paso y la incertidumbre hacen preferible cobrar `+2` pronto.
3. Al comparar el mundo original con uno sin piso resbaloso, cambian las acciones en `(1,2)` y `(3,1)`. El primer cambio es directo; el segundo es un efecto global sobre los valores futuros.
4. El costo `-1` penaliza rutas largas, evita que el robot deambule y favorece alcanzar pronto un terminal.
5. `T(s,a,s')` depende del tipo de piso del estado actual: en piso normal usa `0.90/0.05/0.05`, mientras que en piso resbaloso usa `0.60/0.20/0.20`.

**Predicciones.** Al bajar el costo por paso, la entrega distante se vuelve más atractiva. Al hacer el piso más resbaloso aumenta el riesgo, aunque no necesariamente cambia la acción óptima. Al subir `gamma`, las recompensas futuras pierden menos valor y `+10` gana atractivo. La siguiente celda comprueba estas predicciones.


In [18]:
def intended_path(grid, policy, max_steps=100):
    """Ruta nominal: aplica la dirección elegida sin desviaciones aleatorias."""
    state = grid.start
    path = [state]
    for _ in range(max_steps):
        if grid.is_terminal(state):
            return path
        dr, dc = policy[state]
        candidate = (state[0] + dr, state[1] + dc)
        state = candidate if grid.is_valid_state(candidate) else state
        path.append(state)
        if path.count(state) > 2:
            break
    return path


def solve_experiment(**parameters):
    experiment = WarehouseMDP()
    for name, value in parameters.items():
        setattr(experiment, name, value)
    values, iterations = value_iteration(experiment)
    policy = extract_policy(experiment, values)
    return experiment, values, policy, iterations


experiments = {
    "Base": {},
    "A: living_reward=-0.1": {"living_reward": -0.1},
    "B: resbaloso=0.40": {"slippery_forward_prob": 0.40},
    "C: gamma=0.99": {"gamma": 0.99},
}

for name, parameters in experiments.items():
    experiment, values, policy, iterations = solve_experiment(**parameters)
    path = intended_path(experiment, policy)
    print(f"{name}: {path} (terminal={path[-1]}, iteraciones={iterations})")

# Bonus: busca el primer cambio carga → entrega con precisión de 0.001.
change_at = None
for living_reward in np.arange(-1.0, -0.5, 0.001):
    experiment, _, policy, _ = solve_experiment(living_reward=float(living_reward))
    if intended_path(experiment, policy)[-1] == (0, 5):
        change_at = living_reward
        break

assert change_at is not None
print(f"Bonus: el cambio ocurre aproximadamente en living_reward={change_at:.3f}")


Base: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)] (terminal=(2, 2), iteraciones=20)
A: living_reward=-0.1: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)] (terminal=(0, 5), iteraciones=26)
B: resbaloso=0.40: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)] (terminal=(2, 2), iteraciones=22)
C: gamma=0.99: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)] (terminal=(0, 5), iteraciones=24)
Bonus: el cambio ocurre aproximadamente en living_reward=-0.798
